# [실습1] Tavily Search API 발급 및 사용

## 실습 목표
---
최종 프로젝트에서 웹 검색을 적용하기 위해, Tavily Search API를 발급하고 LangChain을 통해 사용하는 방법을 학습합니다. 

## 실습 목차
---

1. **Tavily Search API 발급:** Tavily AI에 가입하고, API Key를 발급 받습니다.

2. **웹 검색 체인 구성:** 발급 받은 Tavily AI를 활용해 웹 데이터를 검색하고, 그 결과를 바탕으로 답변하는 체인을 구성합니다.

## 실습 개요
---
웹 데이터를 검색하고, 그 결과를 바탕으로 답변하는 체인을 구성하고 사용해봅니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [1]:
import contextlib
import io
import os

import pandas as pd
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_experimental.tools.python.tool import PythonAstREPLTool

- Ollama를 통해 Mistral 7B 모델을 불러옵니다.

In [2]:
# !ollama pull mistral:7b

## 1. Tavily Search API 발급

- Tavily Search API는 LLM과 RAG에 최적화된 웹 검색 API로써, LangChain 공식 튜토리얼에서도 활용하는 검색 API입니다.
- 24년 8월 기준, 별다른 조건 없이 월 1천회 API Call을 무료로 제공합니다. 

1. 먼저, 아래 링크에 접속한 후 'Sign-in' 버튼을 눌러 로그인 화면으로 이동합니다.
   - https://app.tavily.com/sign-in
2. 그 다음, 여러분의 Google, GitHub 계정과 연동되는 계정을 만들거나 아래의 'Sign up' 버튼을 눌러 새로운 계정을 생성합니다.
3. 계정을 생성한 후, 아래 링크에 접속하여 default API Key를 복사하여 아래 코드에 적용합니다.
   - https://app.tavily.com/home

In [3]:
# Tavily API key는 tvly- 로 시작하는 문자열입니다.
# API Key를 입력했다면, 이 셀을 실행해서 API Key를 환경 변수에 등록합니다.
os.environ["TAVILY_API_KEY"] = "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"

API Key를 성공적으로 등록했다면, 아래 코드를 실행해서 Tavily Search API가 잘 작동하는 지 확인합니다.

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_search_tool = TavilySearchResults(max_results=5)

In [5]:
tavily_search_tool.invoke({"query": "삼성 QLED TV 장점"})

[{'url': 'https://m.blog.naver.com/techref/223393094387',
  'content': 'OLED와 QLED 각각의 기술은 사용자의 시청 환경과 용도에 따라 장단점이 있습니다. 예를 들어, 밝은 거실에서 주로 TV를 보는 사람이라면 QLED의 높은 밝기가 유리할 수 있습니다. 반면, 어두운 환경에서 영화를 주로 즐기거나 빠른 반응 속도가 필요한 게임을 즐기는 ...'},
 {'url': 'https://blog.naver.com/PostView.nhn?blogId=modoosmart&logNo=222377894646',
  'content': '우선 TV를 구매하기 앞서 몇 가지 알아야 할 점이 있는데요. OLED QLED 차이점과 4K, 8K TV 화질 비교해서 알려 드려려구요. 1. 삼성 TV 역대 기록. 15년 연속 세계 판매 1위. 8K TV 세계 판매 1위. 초대형 TV 세계 판매 1위. 글로벌 고객만족도 조사 TV 부분 16년 연속 1위. 한국 ...'},
 {'url': 'https://www.samsung.com/sec/tvs/tv-buying-guide/what-is-qled-tv/',
  'content': 'QLED는 튼튼하고 안정적이며 뛰어난 컬러와 밝기를 제공하는 무기물인 퀀텀닷을 사용합니다. OLED는 오래 사용하면 패널이 변질되는 유기물 소재라 동일한 영상이 장기간 반복될 경우. 화면에 잔상이나 얼룩이 영구적으로 남는 번인 현상이 발생할 수 있죠. TV는 한 ...'},
 {'url': 'https://ric4875-1.tistory.com/entry/QLED-OLED-차이-비교',
  'content': '해당 문서는 qled와 oled 차이점과 가격, 장점, 단점을 비교 정리한 문서입니다. qled 또는 oled tv 구매를 고려하시거나, 관련 정보를 얻고자 하시는 분들은 해당 문서를 참조하시길 바랍니다. qled는 삼성의 대표적인 tv 패널이며, oled는 lg의 대표적인 tv 패널입니다.

아래 예시와 비슷한 구조로 출력된다면 정상적으로 API를 적용한 것입니다.
```python
[{'url': 'https://m.blog.naver.com/lifemasterguy/223220367316',
  'content': '삼성 neo qled tv 성능 장점 리뷰 kq85qnc83afxkr kq75qnc83afxkr kq65qnc83afxkr. ... 고급스러운 디자인과 혁신적인 기술력을 자랑하는 삼성의 스마트 qled tv 시리즈입니다. 삼성 neo qled tv는 이미 전 세계적으로 높은 인기를 끌고 있으며, 그 독특한 기능과 성능으로 사용자들 ...'},
 {'url': 'https://m.blog.naver.com/etlandking/220982502313',
  'content': '전자랜드프라이스킹에서 판매 중인 삼성 qled tv, qn65q7famfxkr의 자세한 사양은 아래 링크를 눌러주세요. ... 또 하나의 장점 은 얇은 두께입니다. 이 역시 백라이트와 관련이 있는데요. 자체 발광하는 oled는 별도의 발광 패널이 필요 없기 때문에 상상이상의 얇은 ...'},
 {'url': 'https://www.samsung.com/sec/tvs/qled-qd83afxkr-d2c/KQ75QD83AFXKR/',
  'content': '2024 qled 4k qd83 (189 cm) 제품의 특장점을 확인해 보세요. 2세대 ai 4k 프로세서, 4k ai 업스케일링, 다이렉트 퀀텀 기술을 만나보세요. ... 삼성tv에서는 비밀번호나 핀 코드와 같은 민감한 개인정보가 안전하게 보호됩니다.'},
 {'url': 'https://pocyyy-50.tistory.com/232',
  'content': '다만 LG OLED TV와 직접적인 비교 시 약간 떨어지는 모습을 보입니다. 그러나 삼성 QLED TV에도 단점이 존재합니다. 우선 미니 LED는 LCD를 기반으로 한 패널입니다. 즉 QLED는 마케팅 용어라고 해도 무방합니다. 아무리 LCD를 개량한 미니 LED라고 하지만 OLED 패널 대비 ...'},
 {'url': 'https://m.blog.naver.com/techref/223393094387',
  'content': 'OLED와 QLED 각각의 기술은 사용자의 시청 환경과 용도에 따라 장단점이 있습니다. 예를 들어, 밝은 거실에서 주로 TV를 보는 사람이라면 QLED의 높은 밝기가 유리할 수 있습니다. 반면, 어두운 환경에서 영화를 주로 즐기거나 빠른 반응 속도가 필요한 게임을 즐기는 ...'}]
```

API 적용이 잘 되었다면, 이제 이 검색 결과를 활용해 답변을 생성하는 간단한 체인을 구성해 봅시다.

## 2. 웹 검색 체인 구성

웹 검색 기반 답변 체인은 RAG 기반 답변 체인과 비슷한 구조를 사용해 구현할 수 있습니다.

4챕터 실습3에서 저희는 Vector DB에서 사용자의 질문과 관련이 있는 Document를 Retrieve 했고, 그 텍스트를 이어 붙여서 프롬프트에 증강하였습니다.

이번 실습에서 구현할 웹 체인도 이와 비슷하게, 웹 검색 결과를 이어 붙여서 프롬프트에 증강합니다.


mistral:7b 모델을 사용하는 ChatOllama 객체와 OllamaEmbeddings 객체를 생성합니다.

In [6]:
llm = ChatOllama(model="mistral:7b")
embeddings = OllamaEmbeddings(model="mistral:7b")

다음으로, 사용자의 질문에 대해 Tavily Search API를 통해 여러 문서를 수집하고, 그 결과를 이어 붙이는 함수를 정의합니다.

In [7]:
def tavily_search_and_concat(query: str) -> str:
    results = tavily_search_tool.invoke({"query": query})
    return "\n".join([result["content"] for result in results])

실습2에서 사용한 함수와 유사한 `init_chain()` 함수를 정의합니다.
- 체인에서 `tavily_search_and_concat` 함수를 사용하는 것을 확인할 수 있습니다.

In [8]:
def init_chain():
    messages_with_contexts = [
        ("system", "당신은 마케터를 위한 친절한 지원 챗봇입니다. 웹 검색을 통해 수집한 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]

    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    # Context로 Tavily Serach API 결과를 이어 붙이는 함수를 사용합니다.
    qa_chain = (
        {"context": tavily_search_and_concat, "question": RunnablePassthrough()}
        | prompt_with_context
        | llm
        | StrOutputParser()
    )
    
    return qa_chain

In [9]:
qa_chain = init_chain()

Chain 구성이 완료되었으므로, 사용해봅시다.

In [10]:
question = "삼성 QLED TV 관련 설문조사 결과를 알려줘"

print(qa_chain.invoke(question))

 응답: 수집한 정보에 따르면, 일반 콘텐츠와 HDR(하이다이내믹레인지) 콘텐츠의 화질, 스마트와 게임 기능, 디자인과 연결성 등을 기준으로 한 설문조사 결과에서 참가자들의 약 90%가 삼성 QLED TV를 선호했다고 밝혔습니다. 또한, 영국 소비자 조사기관이 진행한 TV 블라인드 테스트에서 삼성전자의 QLED TV가 '소비자가 선택한 최고 TV'로 선정됐다는 것도 알려졌습니다.


Tavily Search API를 활용해서 저희는 사용자의 질문에 대해 웹에서 정보를 수집해서 이를 바탕으로 답변하는 체인을 구성했습니다. 

최종 프로젝트에서는 이 체인을 활용한 Adaptive RAG 기법을 적용한 삼성전자 카탈로그 기반 마케터 챗봇을 구현해 볼 것입니다.